<a href="https://colab.research.google.com/github/d005810/ECAA08-Manufatura-Flexivel/blob/main/etapa-02-grafos/13%20-%20Algoritmos%20de%20Busca%20BFS%20e%20DFS%20em%20Esteiras%20e%20Triagem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 13 - Notebook: Busca em Largura (BFS) e Busca em Profundidade (DFS) em Esteiras e Triagem
## Célula de Manufatura Flexível (FMS)

Neste notebook implementamos os motores de busca topológica **BFS** (para cálculo determinístico da rota com menor número de atuadores e estágios de esteira) e **DFS** (para enumeração de rotas alternativas de contingência e detecção formal de ciclos via classificação de arestas) aplicados à esteira transportadora e aos postos de triagem pneumática da Célula de Manufatura Flexível.

In [1]:
from collections import deque
from typing import List, Dict, Set, Tuple, Optional, Any

def formatar_tabela(dados: List[Dict[str, Any]]) -> str:
    """Formata lista de dicionários em tabela ASCII pura alinhada."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

def formatar_matriz(matriz: List[List[Any]], rotulos_linhas: List[str], rotulos_cols: List[str]) -> str:
    """Formata matriz 2D em tabela ASCII pura alinhada."""
    larguras = [max(len(str(r)), 7) for r in rotulos_cols]
    larg_linha = max(len(str(r)) for r in rotulos_linhas)
    header = f"{' ' * larg_linha} | " + " | ".join(f"{c:>{larguras[j]}}" for j, c in enumerate(rotulos_cols))
    divisor = f"{'-' * larg_linha}-+-" + "-+-".join("-" * larguras[j] for j in range(len(rotulos_cols)))
    linhas = [header, divisor]
    for i, r_nome in enumerate(rotulos_linhas):
        vals = []
        for j in range(len(rotulos_cols)):
            v = matriz[i][j]
            v_str = "∞" if v == float('inf') else str(v)
            vals.append(f"{v_str:>{larguras[j]}}")
        linhas.append(f"{r_nome:<{larg_linha}} | " + " | ".join(vals))
    return "\n".join(linhas)

class GrafoManufatura:
    """
    Representa a malha de transporte contínuo e postos de triagem da FMS como Dígrafo Ponderado G = (V, E, W).
    """
    def __init__(self, vertices: List[str]):
        self.vertices = vertices
        self.v_to_idx = {v: i for i, v in enumerate(vertices)}
        self.idx_to_v = {i: v for i, v in enumerate(vertices)}
        self.n = len(vertices)

        # Matriz binária de adjacência (1 se há trajeto direcionado u -> v, 0 caso contrário)
        self.adj_binaria = [[0] * self.n for _ in range(self.n)]

        # Matriz de adjacência ponderada por distâncias físicas (metros)
        self.adj_pesos = [[float('inf')] * self.n for _ in range(self.n)]
        for i in range(self.n):
            self.adj_pesos[i][i] = 0.0

        self.arestas_detalhes: List[Dict[str, Any]] = []

    def adicionar_rota(self, origem: str, destino: str, comprimento_m: float,
                        tag_atuador: str, mecanismo: str = "Esteira"):
        """Adiciona um segmento de transporte orientado e parametrizado entre dois postos."""
        u = self.v_to_idx[origem]
        v = self.v_to_idx[destino]
        self.adj_binaria[u][v] = 1
        self.adj_pesos[u][v] = comprimento_m
        self.arestas_detalhes.append({
            "Origem": origem,
            "Destino": destino,
            "Comprimento (m)": comprimento_m,
            "Atuador ISA": tag_atuador,
            "Mecanismo": mecanismo
        })

    def obter_atuador_aresta(self, u: str, v: str) -> str:
        """Retorna a tag ISA do atuador responsável pela transição entre os nós u e v."""
        for a in self.arestas_detalhes:
            if a["Origem"] == u and a["Destino"] == v:
                return a["Atuador ISA"]
        return "N/A"

    def obter_peso_aresta(self, u: str, v: str) -> float:
        """Retorna o comprimento físico do trecho em metros."""
        u_idx = self.v_to_idx[u]
        v_idx = self.v_to_idx[v]
        return self.adj_pesos[u_idx][v_idx]

def criar_planta_fms_nominal() -> GrafoManufatura:
    """Instancia a topologia padrão (DAG linear-ramificado) da Célula de Manufatura Flexível."""
    postos = [
        "SILO-100", "INSP-200", "DIV-201", "DIV-202", "DIV-203",
        "CX-301", "CX-302", "CX-303", "DESC-304"
    ]
    g = GrafoManufatura(postos)
    g.adicionar_rota("SILO-100", "INSP-200", 0.8, "XV-101", "Alimentador Pneumático")
    g.adicionar_rota("INSP-200", "DIV-201",  1.2, "M-101",  "Esteira Principal")
    g.adicionar_rota("DIV-201",  "CX-301",   0.5, "XV-201", "Pistão de Triagem 1")
    g.adicionar_rota("DIV-201",  "DIV-202",  1.0, "M-101",  "Esteira Principal")
    g.adicionar_rota("DIV-202",  "CX-302",   0.5, "XV-202", "Pistão de Triagem 2")
    g.adicionar_rota("DIV-202",  "DIV-203",  1.0, "M-101",  "Esteira Principal")
    g.adicionar_rota("DIV-203",  "CX-303",   0.5, "XV-203", "Pistão de Triagem 3")
    g.adicionar_rota("DIV-203",  "DESC-304", 0.8, "M-101",  "Calha de Descarte")
    return g

def criar_planta_fms_contingencia() -> GrafoManufatura:
    """Instancia a topologia expandida com esteira de bypass e recirculação de refugo."""
    g = criar_planta_fms_nominal()
    # Rota de contingência: Bypass rápido ligando o túnel de inspeção diretamente ao desvio 3
    g.adicionar_rota("INSP-200", "DIV-203", 2.5, "XV-200B", "Bypass de Contingência")
    # Esteira de recirculação: fecha o laço para reaproveitamento de peças descartadas
    g.adicionar_rota("DESC-304", "SILO-100", 3.5, "M-102",  "Esteira de Recirculação")
    return g

class NavegadorFMS:
    """
    Motor de Busca Topológica, Roteamento e Diagnóstico de Intertravamento da FMS.
    """
    def __init__(self, grafo: GrafoManufatura):
        self.g = grafo

    def bfs_menor_caminho(self, origem: str, destino: str,
                          nos_bloqueados: Optional[Set[str]] = None) -> Optional[Dict[str, Any]]:
        """
        Calcula o caminho com menor número de arestas/atuadores comutados via BFS.
        Garante a rota de menor número de saltos graças à invariante da fila FIFO.
        """
        if nos_bloqueados is None:
            nos_bloqueados = set()
        if origem in nos_bloqueados or destino in nos_bloqueados:
            return None

        fila = deque([(origem, [origem])])
        visitados = {origem}

        while fila:
            u_nome, caminho = fila.popleft()
            if u_nome == destino:
                dist_total = 0.0
                atuadores = []
                for i in range(len(caminho) - 1):
                    p_origem = caminho[i]
                    p_dest = caminho[i+1]
                    dist_total += self.g.obter_peso_aresta(p_origem, p_dest)
                    atuadores.append(self.g.obter_atuador_aresta(p_origem, p_dest))
                return {
                    "caminho": caminho,
                    "num_arestas": len(caminho) - 1,
                    "distancia_m": dist_total,
                    "atuadores": atuadores
                }

            u_idx = self.g.v_to_idx[u_nome]
            for v_idx in range(self.g.n):
                v_nome = self.g.idx_to_v[v_idx]
                if (self.g.adj_binaria[u_idx][v_idx] == 1 and
                    v_nome not in visitados and
                    v_nome not in nos_bloqueados):
                    visitados.add(v_nome)
                    fila.append((v_nome, caminho + [v_nome]))

        return None

    def dfs_todos_os_caminhos(self, origem: str, destino: str,
                             nos_bloqueados: Optional[Set[str]] = None) -> List[List[str]]:
        """
        Enumera todas as rotas possíveis entre origem e destino via DFS com retrocesso (backtracking).
        """
        if nos_bloqueados is None:
            nos_bloqueados = set()
        if origem in nos_bloqueados or destino in nos_bloqueados:
            return []

        caminhos: List[List[str]] = []

        def dfs(u_nome: str, visitados: List[str]):
            if u_nome == destino:
                caminhos.append(list(visitados))
                return
            u_idx = self.g.v_to_idx[u_nome]
            for v_idx in range(self.g.n):
                v_nome = self.g.idx_to_v[v_idx]
                if (self.g.adj_binaria[u_idx][v_idx] == 1 and
                    v_nome not in visitados and
                    v_nome not in nos_bloqueados):
                    visitados.append(v_nome)
                    dfs(v_nome, visitados)
                    visitados.pop()

        dfs(origem, [origem])
        return caminhos

    def dfs_classificar_arestas(self, origem: str) -> Dict[str, Any]:
        """
        Executa DFS com marcação temporal e classifica cada aresta em Árvore, Retorno, Avanço ou Cruzamento.
        Comprova formalmente se a malha é um DAG (ausência de arestas de retorno).
        """
        BRANCO, CINZA, PRETO = 0, 1, 2
        cores = {v: BRANCO for v in self.g.vertices}
        d_tempo = {v: 0 for v in self.g.vertices}
        f_tempo = {v: 0 for v in self.g.vertices}
        tempo = 0
        arestas_classificadas = []
        arestas_retorno = []

        def dfs_visita(u: str):
            nonlocal tempo
            tempo += 1
            d_tempo[u] = tempo
            cores[u] = CINZA

            u_idx = self.g.v_to_idx[u]
            for v_idx in range(self.g.n):
                if self.g.adj_binaria[u_idx][v_idx] == 1:
                    v = self.g.idx_to_v[v_idx]
                    aresta_str = f"{u} -> {v}"
                    if cores[v] == BRANCO:
                        arestas_classificadas.append({
                            "Aresta": aresta_str,
                            "Tipo": "Árvore (Tree Edge)",
                            "Diagnóstico": "Segmento Principal"
                        })
                        dfs_visita(v)
                    elif cores[v] == CINZA:
                        arestas_classificadas.append({
                            "Aresta": aresta_str,
                            "Tipo": "Retorno (Back Edge)",
                            "Diagnóstico": "CICLO DETECTADO (Recirculação)"
                        })
                        arestas_retorno.append(aresta_str)
                    elif d_tempo[u] < d_tempo[v]:
                        arestas_classificadas.append({
                            "Aresta": aresta_str,
                            "Tipo": "Avanço (Forward Edge)",
                            "Diagnóstico": "Bypass / Atalho"
                        })
                    else:
                        arestas_classificadas.append({
                            "Aresta": aresta_str,
                            "Tipo": "Cruzamento (Cross Edge)",
                            "Diagnóstico": "Conexão Inter-Ramos"
                        })

            cores[u] = PRETO
            tempo += 1
            f_tempo[u] = tempo

        dfs_visita(origem)
        for v in self.g.vertices:
            if cores[v] == BRANCO:
                dfs_visita(v)

        return {
            "arestas": arestas_classificadas,
            "arestas_retorno": arestas_retorno,
            "eh_dag": len(arestas_retorno) == 0,
            "d_tempo": d_tempo,
            "f_tempo": f_tempo
        }

    def auditar_conectividade_scada(self, origem: str, destinos: List[str],
                                    nos_bloqueados: Optional[Set[str]] = None) -> List[Dict[str, Any]]:
        """
        Auditoria de conectividade global em tempo de varredura para o supervisório SCADA.
        """
        if nos_bloqueados is None:
            nos_bloqueados = set()

        relatorio = []
        for dst in destinos:
            res_bfs = self.bfs_menor_caminho(origem, dst, nos_bloqueados)
            if res_bfs is not None:
                rota_str = " -> ".join(res_bfs["caminho"])
                atuadores_str = ", ".join(res_bfs["atuadores"])
                status = "OPERACIONAL"
                if "XV-200B" in res_bfs["atuadores"]:
                    status = "CONTINGÊNCIA (Bypass)"
                relatorio.append({
                    "Destino": dst,
                    "Status SCADA": status,
                    "Saltos (Arestas)": res_bfs["num_arestas"],
                    "Distância (m)": f"{res_bfs['distancia_m']:.1f}",
                    "Rota Ativa": rota_str,
                    "Atuadores Ativados": atuadores_str
                })
            else:
                relatorio.append({
                    "Destino": dst,
                    "Status SCADA": "BLOQUEADO (Intertravamento)",
                    "Saltos (Arestas)": "N/A",
                    "Distância (m)": "N/A",
                    "Rota Ativa": "Nenhum caminho viável",
                    "Atuadores Ativados": "INIBIÇÃO DE INFEED"
                })
        return relatorio

# --- INSTANCIAÇÃO DOS MODELOS TOPOLÓGICOS ---
fms_nominal = criar_planta_fms_nominal()
fms_contingencia = criar_planta_fms_contingencia()

nav_nominal = NavegadorFMS(fms_nominal)
destinos_fms = ["CX-301", "CX-302", "CX-303", "DESC-304"]

# 1. Roteamento Nominal BFS para cada posto de destino
print("=== 1. AUDITORIA DE CONECTIVIDADE SCADA - OPERAÇÃO NOMINAL DA FMS ===")
relatorio_nominal = nav_nominal.auditar_conectividade_scada("SILO-100", destinos_fms)
print(formatar_tabela(relatorio_nominal))

# 2. Simulação de Falha com Bloqueio Dinâmico (DIV-202 bloqueado por falha no pistão XV-202)
print("\n=== 2. SIMULAÇÃO DE FALHA: PISTÃO DE DESVIO 2 COM DEFEITO (DIV-202 BLOQUEADO) ===")
relatorio_falha = nav_nominal.auditar_conectividade_scada("SILO-100", destinos_fms, nos_bloqueados={"DIV-202"})
print(formatar_tabela(relatorio_falha))

# 3. Malha de Contingência com Bypass e Recirculação Ativados
nav_contingencia = NavegadorFMS(fms_contingencia)
print("\n=== 3. MALHA DE CONTINGÊNCIA ATIVADA (BYPASS INSP-200 -> DIV-203) COM DIV-202 BLOQUEADO ===")
relatorio_cont = nav_contingencia.auditar_conectividade_scada("SILO-100", destinos_fms, nos_bloqueados={"DIV-202"})
print(formatar_tabela(relatorio_cont))

# 4. Enumeração DFS de todas as opções de rota para CX-303 (Malha expandida)
rotas_cx3 = nav_contingencia.dfs_todos_os_caminhos("SILO-100", "CX-303")
print(f"\n=== 4. ENUMERAÇÃO DE TODAS AS ROTAS PARA CX-303 VIA DFS (MALHA EXPANDIDA) ===")
print(f"Total de opções de rotas encontradas: {len(rotas_cx3)}")
for idx, rota in enumerate(rotas_cx3, 1):
    num_saltos = len(rota) - 1
    tipo_rota = "Bypass Auxiliar" if "DIV-203" in rota and "DIV-201" not in rota else "Rota Nominal"
    print(f"  Opção {idx} ({tipo_rota}, {num_saltos} saltos): {' -> '.join(rota)}")

# 5. Classificação Formal de Arestas DFS e Diagnóstico Estrutural de DAG
print("\n=== 5. CLASSIFICAÇÃO FORMAL DE ARESTAS VIA DFS E AUDITORIA DE DAG ===")
diag_nominal = nav_nominal.dfs_classificar_arestas("SILO-100")
print("\n-- TOPOLOGIA NOMINAL DA FMS:")
print(formatar_tabela(diag_nominal["arestas"]))
print(f"Arestas de Retorno (Back Edges) detectadas: {len(diag_nominal['arestas_retorno'])}")
print(f"Diagnóstico Estrutural: {'A malha nominal é estritamente um DÍGRAFO ACÍCLICO DIRIGIDO (DAG).' if diag_nominal['eh_dag'] else 'Ciclos detectados!'}")

diag_cont = nav_contingencia.dfs_classificar_arestas("SILO-100")
print("\n-- TOPOLOGIA COM ESTEIRA DE RECIRCULAÇÃO (DESC-304 -> SILO-100):")
print(f"Arestas de Retorno (Back Edges) detectadas: {len(diag_cont['arestas_retorno'])}")
for ar in diag_cont["arestas_retorno"]:
    print(f"  -> {ar}: Retorno (Back Edge) [CICLO DETECTADO (Recirculação)]")
print(f"Diagnóstico Estrutural: {'DAG confirmado.' if diag_cont['eh_dag'] else 'Topologia em MALHA FECHADA identificada. Loop de recirculação ativo!'}")

# --- 6. ASSERÇÕES DETERMINÍSTICAS DE VALIDAÇÃO ---
res_cx1 = nav_nominal.bfs_menor_caminho("SILO-100", "CX-301")
assert res_cx1 is not None and res_cx1["num_arestas"] == 3
assert res_cx1["caminho"] == ["SILO-100", "INSP-200", "DIV-201", "CX-301"]

res_cx2 = nav_nominal.bfs_menor_caminho("SILO-100", "CX-302")
assert res_cx2 is not None and res_cx2["num_arestas"] == 4

res_cx3 = nav_nominal.bfs_menor_caminho("SILO-100", "CX-303")
assert res_cx3 is not None and res_cx3["num_arestas"] == 5

# Validações de bloqueio sob falha
assert nav_nominal.bfs_menor_caminho("SILO-100", "CX-302", nos_bloqueados={"DIV-202"}) is None
assert nav_nominal.bfs_menor_caminho("SILO-100", "CX-301", nos_bloqueados={"DIV-202"}) is not None

# Validações de DAG e classificação
assert diag_nominal["eh_dag"] is True
assert len(diag_nominal["arestas_retorno"]) == 0
assert diag_cont["eh_dag"] is False
assert "DESC-304 -> SILO-100" in diag_cont["arestas_retorno"]
assert len(rotas_cx3) == 2

print("\n[OK] Todos os algoritmos de busca BFS/DFS e diagnósticos de integridade validados com sucesso!")

=== 1. AUDITORIA DE CONECTIVIDADE SCADA - OPERAÇÃO NOMINAL DA FMS ===
Destino  | Status SCADA | Saltos (Arestas) | Distância (m) | Rota Ativa                                                        | Atuadores Ativados                         
---------+--------------+------------------+---------------+-------------------------------------------------------------------+--------------------------------------------
CX-301   | OPERACIONAL  | 3                | 2.5           | SILO-100 -> INSP-200 -> DIV-201 -> CX-301                         | XV-101, M-101, XV-201                      
CX-302   | OPERACIONAL  | 4                | 3.5           | SILO-100 -> INSP-200 -> DIV-201 -> DIV-202 -> CX-302              | XV-101, M-101, M-101, XV-202               
CX-303   | OPERACIONAL  | 5                | 4.5           | SILO-100 -> INSP-200 -> DIV-201 -> DIV-202 -> DIV-203 -> CX-303   | XV-101, M-101, M-101, M-101, XV-203        
DESC-304 | OPERACIONAL  | 5                | 4.8           | SILO